In [2]:
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy, \
    StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os

from langchain_core.runnables import RunnableConfig
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
    reasoning_effort="none"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
@dataclass
class ContractInfo:
    """
    提取用户的联系方式
    """
    name:str=Field(description="用户姓名")
    email:str=Field(description="用户邮箱号")
    phone:str=Field(description="用户手机号")
@dataclass
class EventInfo:
    """
    事件详情
    """
    event_name:str=Field(description="事件名称")
    date:str=Field(description="事件发送日期")



{
    'messages': [
        HumanMessage(
            content='请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333;日期：2026年7月9
日，名称：高考人数已经突破1200万',
            additional_kwargs={},
            response_metadata={},
            id='5fccfb88-cc7e-4336-9cc4-34350031c00c'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 66,
                    'prompt_tokens': 492,
                    'total_tokens': 558,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 492
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-d66b00d7-d883-9d19-9893-1cc5ebffd74c',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a0aa81-fd70-75d1-a312-2de2d5ad8445-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': '1414227322@qq.com', 'phone': '18788483333'},
                    'id': 'call_d71bfd9967a84f95897a2282',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 492,
                'output_tokens': 66,
                'total_tokens': 558,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='已成功抽取结构信息',
            name='ContractInfo',
            id='6d3c23fe-d4bc-49ac-83d9-39088e570c01',
            tool_call_id='call_d71bfd9967a84f95897a2282'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='1414227322@qq.com', phone='18788483333')
}

1 handle_errors=True 默认方式，捕获所有异常，有内置的错误消息模板，适用于大多出自动处理错误的通用场景

In [6]:
#3 初始化我的agent
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    # system_prompt="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。"
    system_prompt=SystemMessage(content="你是1个专业的信息处理小助手,当有人问你关于信息处理问题的时候，你会帮助询问者完成信息处理任务。"),
    response_format=ToolStrategy(
        schema=Union[ContractInfo,EventInfo],
        tool_message_content="已成功抽取结构信息",
        handle_errors=True#默认值是True
    ),
)
#4 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [                HumanMessage("请你提取以下文本的机构信息：小明的邮箱是1414227322@qq.com,手机是18788483333;日期：2026年7月9日，名称：高考人数已经突破1200万")        ]
    }
)
rprint(response)

{
    'messages': [
        HumanMessage(
            content='请你提取以下文本的机构信息：小明的邮箱是1414227322@qq.com,手机是18788483333;日期：2026年7月9日
，名称：高考人数已经突破1200万',
            additional_kwargs={},
            response_metadata={},
            id='c9d0882d-6338-441e-9c24-28223722c136'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 120,
                    'prompt_tokens': 492,
                    'total_tokens': 612,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 492
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-d8ecf8c1-b231-9c4c-beef-b8a1d0f541cf',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a0aa85-035a-7450-a697-c096614eeb77-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': '1414227322@qq.com', 'phone': '18788483333'},
                    'id': 'call_c0d08a1e9ac242fd9e33955b',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventInfo',
                    'args': {'event_name': '高考人数已经突破1200万', 'date': '2026年7月9日'},
                    'id': 'call_7dac770852284eacbc2924f6',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 492,
                'output_tokens': 120,
                'total_tokens': 612,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContractInfo, EventInfo) when
only one is expected.\n Please fix your mistakes.',
            name='ContractInfo',
            id='20f33402-076b-4531-aa6f-1f9efa6fcb79',
            tool_call_id='call_c0d08a1e9ac242fd9e33955b'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContractInfo, EventInfo) when
only one is expected.\n Please fix your mistakes.',
            name='EventInfo',
            id='fcfe0dc8-a6c5-468e-9c44-3585a9560bc9',
            tool_call_id='call_7dac770852284eacbc2924f6'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 66,
                    'prompt_tokens': 692,
                    'total_tokens': 758,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 692
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-b9f291e9-a7c5-9a44-a44e-ce41b615e67e',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a0aa85-0acd-7f90-b506-f078c7585382-0',
            tool_calls=[
                {
       

2 handle_errors=False 关闭自动重试机制，会直接抛出异常然后终止程序

In [7]:
#3 初始化我的agent
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    # system_prompt="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。"
    system_prompt=SystemMessage(content="你是1个专业的信息处理小助手,当有人问你关于信息处理问题的时候，你会帮助询问者完成信息处理任务。"),
    response_format=ToolStrategy(
        schema=Union[ContractInfo,EventInfo],
        tool_message_content="已成功抽取结构信息",
        handle_errors=False#默认值是True
    ),
)
#4 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [                HumanMessage("请你提取以下文本的结构信息：小明的邮箱是1414227322@qq.com,手机是18788483333;日期：2026年7月9日，名称：高考人数已经突破1200万")        ]
    }
)
rprint(response)

MultipleStructuredOutputsError: Model incorrectly returned multiple structured responses (ContractInfo, EventInfo) when only one is expected.

3 handle_errors="自定义字符串"  捕获所有异常，但是使用开发者预设的错误消息模板，适用于特定业务引导的场景

In [12]:
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union
from langchain.agents.structured_output import ProviderStrategy,ToolStrategy,AutoStrategy
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=1,
    reasoning_effort="none"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
@dataclass
class ContractInfo:
    """
    提取用户的联系方式
    """
    name:str=Field(description="用户姓名")
    email:str=Field(description="用户邮箱号")
    phone:str=Field(description="用户手机号")
@dataclass
class EventInfo:
    """
    事件详情
    """
    event_name:str=Field(description="事件名称")
    date:str=Field(description="事件发送日期")

#3 初始化我的agent
myAgent1=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    # system_prompt="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。"
    system_prompt=SystemMessage(content="你是1个专业的信息处理小助手,当有人问你关于信息处理问题的时候，你会帮助询问者完成信息处理任务。"),
    response_format=ToolStrategy(
        schema=Union[ContractInfo,EventInfo],
        tool_message_content="已成功抽取结构信息",
        handle_errors="请检查输入的数据"#默认值是True
    ),
)

#4 调用agent
response1=myAgent1.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [                HumanMessage("请你提取以下文本的结构信息：小明的邮箱是1414227322@qq.com,手机是18788483333;日期：2026年7月9日，名称：高考人数已经突破1200万")        ]
    }
)
rprint(response1)

OpenAIInvalidRequestError: Error code: 400 - {'error': {'code': 'invalid_parameter_error', 'param': None, 'message': 'Repetitive tool calls detected in the conversation history. The same tool call with identical name and arguments has been repeated across multiple consecutive rounds. Please modify your request or adjust the tool call arguments to avoid infinite loops.', 'type': 'invalid_request_error'}, 'id': 'chatcmpl-50a18785-dd21-9beb-9175-c475182e8def'}

4 handle_errors=ExceptionType  捕获特定异常，其他异常直接抛出，适用于需要精准控制，只针对特定错误进行重试和排查的场景

In [14]:
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union
from langchain.agents.structured_output import ProviderStrategy,ToolStrategy,AutoStrategy,MultipleStructuredOutputsError,StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=1,
    reasoning_effort="none"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
@dataclass
class ContractInfo:
    """
    提取用户的联系方式
    """
    name:str=Field(description="用户姓名")
    email:str=Field(description="用户邮箱号")
    phone:str=Field(description="用户手机号")
@dataclass
class EventInfo:
    """
    事件详情
    """
    event_name:str=Field(description="事件名称")
    date:str=Field(description="事件发送日期")

#3 初始化我的agent
myAgent1=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    # system_prompt="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。"
    system_prompt=SystemMessage(content="你是1个专业的信息处理小助手,当有人问你关于信息处理问题的时候，你会帮助询问者完成信息处理任务。"),
    response_format=ToolStrategy(
        schema=Union[ContractInfo,EventInfo],
        tool_message_content="已成功抽取结构信息",
        # handle_errors=MultipleStructuredOutputsError#默认值是True
        handle_errors=StructuredOutputValidationError#默认值是True
    ),
)
#4 调用agent
response1=myAgent1.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [                HumanMessage("请你提取以下文本的结构信息：小明的邮箱是1414227322@qq.com,手机是18788483333;日期：2026年7月9日，名称：高考人数已经突破1200万")        ]
    }
)
rprint(response1)

MultipleStructuredOutputsError: Model incorrectly returned multiple structured responses (ContractInfo, EventInfo) when only one is expected.

5 handle_errors=callable  灵活性最高的方式，使用开发者自定义函数来处理异常，适用于需要复杂、精细化错误处理的场景

In [15]:
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union
from langchain.agents.structured_output import ProviderStrategy,ToolStrategy,AutoStrategy,MultipleStructuredOutputsError,StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=1,
    reasoning_effort="none"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
@dataclass
class ContractInfo:
    """
    提取用户的联系方式
    """
    name:str=Field(description="用户姓名")
    email:str=Field(description="用户邮箱号")
    phone:str=Field(description="用户手机号")
@dataclass
class EventInfo:
    """
    事件详情
    """
    event_name:str=Field(description="事件名称")
    date:str=Field(description="事件发送日期")

#3 自定义1个错误处理函数
def custom_error_handler(error:Exception) ->str:
    """自定义错误处理器"""
    error_str=str(error)
    print(f"捕获到的错误类型是：{type(error_str).__name__}")
    print(f"错误详情是：{error_str}")
    if isinstance(error,MultipleStructuredOutputsError,):
        return"检查到多个响应，请选择最相关的"
    elif isinstance(error,StructuredOutputValidationError):
        return "数据格式有误，请检查字段是否符合要求"
    else:
        return f"Error{error_str}"
#4 初始化我的agent
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    # system_prompt="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。"
    system_prompt=SystemMessage(content="你是1个专业的信息处理小助手,当有人问你关于信息处理问题的时候，你会帮助询问者完成信息处理任务。"),
    response_format=ToolStrategy(
        schema=Union[ContractInfo,EventInfo],
        tool_message_content="已成功抽取结构信息",
        # handle_errors=MultipleStructuredOutputsError#默认值是True
        handle_errors=custom_error_handler#默认值是True
    ),
)

#5 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [                HumanMessage("请你提取以下文本的结构信息：小明的邮箱是1414227322@qq.com,手机是18788483333;日期：2026年7月9日，名称：高考人数已经突破1200万")        ]
    }
)
rprint(response)

捕获到的错误类型是：str
错误详情是：Model incorrectly returned multiple structured responses (ContractInfo, EventInfo) when only one is expected.
捕获到的错误类型是：str
错误详情是：Model incorrectly returned multiple structured responses (ContractInfo, EventInfo) when only one is expected.
捕获到的错误类型是：str
错误详情是：Model incorrectly returned multiple structured responses (ContractInfo, EventInfo) when only one is expected.
捕获到的错误类型是：str
错误详情是：Model incorrectly returned multiple structured responses (ContractInfo, EventInfo) when only one is expected.
捕获到的错误类型是：str
错误详情是：Model incorrectly returned multiple structured responses (ContractInfo, EventInfo) when only one is expected.
捕获到的错误类型是：str
错误详情是：Model incorrectly returned multiple structured responses (ContractInfo, EventInfo) when only one is expected.
捕获到的错误类型是：str
错误详情是：Model incorrectly returned multiple structured responses (ContractInfo, EventInfo) when only one is expected.
捕获到的错误类型是：str
错误详情是：Model incorrectly returned multiple structured responses (ContractInfo

OpenAIInvalidRequestError: Error code: 400 - {'error': {'code': 'invalid_parameter_error', 'param': None, 'message': 'Repetitive tool calls detected in the conversation history. The same tool call with identical name and arguments has been repeated across multiple consecutive rounds. Please modify your request or adjust the tool call arguments to avoid infinite loops.', 'type': 'invalid_request_error'}, 'id': 'chatcmpl-0cd53039-a501-971b-a061-d0266b59d826'}

设置agent实例的最大重试次数，避免多次无效循环

In [16]:
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union
from langchain.agents.structured_output import ProviderStrategy,ToolStrategy,AutoStrategy,MultipleStructuredOutputsError,StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=1,
    reasoning_effort="none"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
@dataclass
class ContractInfo:
    """
    提取用户的联系方式
    """
    name:str=Field(description="用户姓名")
    email:str=Field(description="用户邮箱号")
    phone:str=Field(description="用户手机号")
@dataclass
class EventInfo:
    """
    事件详情
    """
    event_name:str=Field(description="事件名称")
    date:str=Field(description="事件发送日期")

#3 自定义1个错误处理函数
def custom_error_handler(error:Exception) ->str:
    """自定义错误处理器"""
    error_str=str(error)
    print(f"捕获到的错误类型是：{type(error_str).__name__}")
    print(f"错误详情是：{error_str}")
    if isinstance(error,MultipleStructuredOutputsError,):
        return"检查到多个响应，请选择最相关的"
    elif isinstance(error,StructuredOutputValidationError):
        return "数据格式有误，请检查字段是否符合要求"
    else:
        return f"Error{error_str}"
#4 设置最大响应次数的配置文件，避免多次无效循环
my_config=RunnableConfig(
    recursion_limit=3
)
#4 初始化我的agent
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    # system_prompt="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。"
    system_prompt=SystemMessage(content="你是1个专业的信息处理小助手,当有人问你关于信息处理问题的时候，你会帮助询问者完成信息处理任务。"),
    response_format=ToolStrategy(
        schema=Union[ContractInfo,EventInfo],
        tool_message_content="已成功抽取结构信息",
        # handle_errors=MultipleStructuredOutputsError#默认值是True
        handle_errors=custom_error_handler#默认值是True
    ),
)

#5 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [                HumanMessage("请你提取以下文本的结构信息：小明的邮箱是1414227322@qq.com,手机是18788483333;日期：2026年7月9日，名称：高考人数已经突破1200万")        ]
    },
    config=my_config,
)
rprint(response)

捕获到的错误类型是：str
错误详情是：Model incorrectly returned multiple structured responses (ContractInfo, EventInfo) when only one is expected.
捕获到的错误类型是：str
错误详情是：Model incorrectly returned multiple structured responses (ContractInfo, EventInfo) when only one is expected.
捕获到的错误类型是：str
错误详情是：Model incorrectly returned multiple structured responses (ContractInfo, EventInfo) when only one is expected.


GraphRecursionError: Recursion limit of 3 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT